In [1]:
!pip install transformers datasets sentencepiece accelerate underthesea

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 97.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 78.0 MB/s eta 0:00:00


In [2]:
!unzip /content/drive/MyDrive/TrainAi_translate_app/colab_dataset_ede.zip

Archive:  /content/drive/MyDrive/TrainAi_translate_app/colab_dataset_ede.zip
   creating: colab_dataset_ede/
  inflating: colab_dataset_ede/test_data.json  
  inflating: colab_dataset_ede/train_data.json  
  inflating: colab_dataset_ede/valid_data.json  


In [3]:
import json
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

# 1. Load Dataset
def load_json(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        return json.load(f)

train_data = load_json('colab_dataset_ede/train_data.json')
valid_data = load_json('colab_dataset_ede/valid_data.json')

# Biến đổi thành HuggingFace Dataset
train_dataset = Dataset.from_list(train_data)
valid_dataset = Dataset.from_list(valid_data)

datasets = DatasetDict({
    "train": train_dataset,
    "validation": valid_dataset
})

# 2. Load Tokenizer & Model (vinai/bartpho-word)
model_checkpoint = "vinai/bartpho-word"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

# Đóng băng Encoder để tăng tốc độ huấn luyện giống bài báo
for param in model.model.encoder.parameters():
    param.requires_grad = False

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/897 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.68G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [4]:
max_length = 256

def preprocess_function(examples):
    inputs = [ex for ex in examples["vi"]]
    targets = [ex for ex in examples["ede"]]

    model_inputs = tokenizer(inputs, max_length=max_length, truncation=True)

   # Setup the tokenizer for targets
    labels = tokenizer(text_target=targets, max_length=max_length, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = datasets.map(preprocess_function, batched=True)

Map:   0%|          | 0/15092 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [5]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir="./bartpho-ede-nmt",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=15,
    predict_with_generate=True,
    fp16=True, # Dùng Mixed Precision để train nhanh hơn trên GPU
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
)

# Bắt đầu huấn luyện
trainer.train()

Epoch,Training Loss,Validation Loss
1,2.834922,1.796011
2,1.744241,1.487096
3,1.430085,1.353008
4,1.247549,1.265004
5,1.103256,1.217247
6,1.013102,1.185958
7,0.923745,1.170959
8,0.853623,1.158662
9,0.798850,1.153468
10,0.738113,1.154982


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=14160, training_loss=1.0312009806013376, metrics={'train_runtime': 6219.9907, 'train_samples_per_second': 36.396, 'train_steps_per_second': 2.277, 'total_flos': 2.8197481995239424e+16, 'train_loss': 1.0312009806013376, 'epoch': 15.0})

In [6]:
# Đường dẫn tới thư mục trên Google Drive của bạn
drive_save_path = "/content/drive/MyDrive/TrainAi_translate_app/best-vntoede-model"

# Lưu trực tiếp mô hình vào Drive
trainer.save_model(drive_save_path)

# Nén mô hình thành file zip và cũng lưu luôn trên Drive (nếu bạn muốn tải về máy cá nhân cho gọn)
!zip -r /content/drive/MyDrive/TrainAi_translate_app/best-vntoede-model.zip {drive_save_path}

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  adding: content/drive/MyDrive/TrainAi_translate_app/best-vntoede-model/ (stored 0%)
  adding: content/drive/MyDrive/TrainAi_translate_app/best-vntoede-model/config.json (deflated 58%)
  adding: content/drive/MyDrive/TrainAi_translate_app/best-vntoede-model/generation_config.json (deflated 41%)
  adding: content/drive/MyDrive/TrainAi_translate_app/best-vntoede-model/model.safetensors (deflated 24%)
  adding: content/drive/MyDrive/TrainAi_translate_app/best-vntoede-model/tokenizer_config.json (deflated 76%)
  adding: content/drive/MyDrive/TrainAi_translate_app/best-vntoede-model/added_tokens.json (stored 0%)
  adding: content/drive/MyDrive/TrainAi_translate_app/best-vntoede-model/vocab.txt (deflated 55%)
  adding: content/drive/MyDrive/TrainAi_translate_app/best-vntoede-model/bpe.codes (deflated 59%)
  adding: content/drive/MyDrive/TrainAi_translate_app/best-vntoede-model/training_args.bin (deflated 53%)
